# Fase 2 — Perbandingan Model Peramalan Multi-Horizon

**Tujuan.** Menguji hipotesis yang diturunkan Fase 1: apakah variabel yang
menolak hipotesis jalan acak benar-benar dapat diramalkan lebih baik daripada
model dasar, dan apakah jaringan saraf memberi perbaikan di atas model linear.

**Sembilan model** dibandingkan pada protokol yang identik:

| Keluarga | Model |
|---|---|
| Dasar | Naive, Seasonal Naive |
| Statistik | Linear (autoregresif), ARIMA |
| Ansambel pohon | Random Forest |
| Jaringan saraf | MLP, DNN, LSTM, GRU |

**Lima horizon**: 1, 3, 6, 12, 24 langkah — setara 5 menit hingga 2 jam.

**Yang membuat perbandingan ini sah:**

1. **Seasonal Naive disertakan** supaya keunggulan model tidak bisa dijelaskan
   sekadar sebagai "mempelajari siklus harian".
2. **Multi-seed** untuk jaringan saraf (3 benih) — hasil dilaporkan sebagai rerata,
   sebab satu kali pelatihan dapat menyesatkan.
3. **Uji Diebold-Mariano** menentukan apakah selisih MAE bermakna atau kebetulan.
4. **Bootstrap blok** memberi selang kepercayaan yang menghormati autokorelasi.

> **Penting.** Selisih MAE yang kecil **bukan bukti keunggulan**. Hanya uji
> Diebold-Mariano yang boleh dipakai menyimpulkan satu model lebih baik.

Waktu jalan: ±10 menit di CPU Colab, lebih cepat dengan GPU.

In [ ]:
!pip install -q statsmodels scikit-learn pandas numpy scipy matplotlib
print('siap')

## 1. Muat dataset beku dan verifikasi

Sama seperti Fase 1: berkas diambil dari repositori lalu SHA-256-nya dicocokkan,
sehingga hasil dapat dipastikan berasal dari data yang sama persis.

In [ ]:
import io, os, math, json, time, hashlib, warnings, itertools
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt
from scipy import stats as sstats

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

URL = 'https://raw.githubusercontent.com/jejenfisika-stack/IoT-Kopi_Web3/main/dataset/sesi_20260824-20260829.csv'
SHA_HARAP = '92025679db8c2ab50123ebe812ace209f60643cd0c229ac16aca5e1c44454690'
OUT = 'hasil_fase2'; os.makedirs(OUT, exist_ok=True)

RULE       = '5min'
VAR        = {'suhu': ('Suhu udara', '\u00b0C'),
              'udara': ('Kelembaban udara', '%'),
              'tanah': ('Kelembaban tanah', '%')}
HORIZON    = [1, 3, 6, 12, 24]   # 5 menit .. 2 jam
W          = 12                   # 12 x 5 menit = 1 jam riwayat
MUSIM      = 288                  # 1 hari pada resolusi 5 menit
TRAIN_FRAC = 0.70
MAKS_ORIGIN= 150
SEEDS      = [42, 7, 123]
EPOCHS     = 60
MAD_K      = 4.0
N_BOOT     = 1000
MODEL_DL   = ['MLP', 'DNN', 'LSTM', 'GRU']   # kosongkan [] untuk jalan cepat tanpa DL

T0 = time.time()
def log(*a): print(f'[{time.time()-T0:6.1f}s]', *a, flush=True)

mentah = requests.get(URL, timeout=180).content
sha = hashlib.sha256(mentah).hexdigest()
print('SHA-256 cocok :', sha == SHA_HARAP)
df = pd.read_csv(io.BytesIO(mentah), parse_dates=['waktu']).set_index('waktu').sort_index()
print(f'bacaan {len(df)} | {df.index.min()} -> {df.index.max()}')

## 2. Pra-pemrosesan

Deret di-*resample* ke 5 menit, lalu pencilan disaring dengan **Median Absolute
Deviation** (ambang 4·MAD) — kokoh terhadap sebaran berekor berat, berbeda dari
simpangan baku yang justru terseret oleh pencilan itu sendiri.

Celah yang tersisa diisi interpolasi. Jumlah titik yang dibuang dilaporkan
supaya dampaknya dapat dinilai pembaca.

In [ ]:
def siapkan(kolom):
    s = df[kolom].resample(RULE).mean()
    kosong = int(s.isna().sum())
    med = s.median(); mad = (s - med).abs().median()
    n_pencilan = 0
    if pd.notna(mad) and mad > 0:
        topeng = (s - med).abs() > MAD_K * mad
        n_pencilan = int(topeng.sum()); s[topeng] = np.nan
    return s.interpolate(limit_direction='both').dropna(), kosong, n_pencilan

seri, lap = {}, []
for k, (nama, sat) in VAR.items():
    s, kosong, pencilan = siapkan(k)
    seri[k] = s
    lap.append({'Variabel': nama, 'Titik': len(s), 'Bin kosong': kosong,
                'Pencilan dibuang': pencilan,
                'Dibuang (%)': round(100*pencilan/max(1, len(s)), 2)})
tab_pra = pd.DataFrame(lap)
print('TABEL 1 — Pra-pemrosesan'); display(tab_pra)

## 3. Protokol evaluasi

**Pembagian kronologis** 70% latih / 30% uji — tidak diacak, sebab mengacak deret
waktu membocorkan masa depan ke masa lalu.

**Rolling origin**: dari banyak titik asal di bagian uji, tiap model meramal
1–24 langkah ke depan **tanpa pernah melihat nilai sebenarnya**. Prediksi langkah
sebelumnya dipakai sebagai masukan langkah berikutnya (rekursif).

Model dilatih **sekali** pada bagian latih, lalu dievaluasi pada seluruh titik
asal. Melatih ulang di tiap origin akan lebih ketat tetapi ratusan kali lebih
mahal, dan tidak lazim pada perbandingan skala ini.

Rekursi **divektorkan untuk semua origin sekaligus** sehingga 150 origin × 24
langkah hanya butuh 24 pemanggilan model, bukan 3.600.

In [ ]:
def buat_origin(n, n_train, h_maks):
    mulai = max(n_train, MUSIM + W)
    semua = list(range(mulai, n - h_maks))
    if len(semua) > MAKS_ORIGIN:
        langkah = len(semua) / MAKS_ORIGIN
        semua = [semua[int(i*langkah)] for i in range(MAKS_ORIGIN)]
    return semua

def jendela(y, origins):
    return np.array([y[o-W:o] for o in origins], dtype=float)

def rekursif(prediksi_satu_langkah, X0, h):
    """Ramal h langkah rekursif, divektorkan untuk SEMUA origin sekaligus."""
    X = X0.copy(); keluar = np.zeros((X0.shape[0], h))
    for t in range(h):
        p = prediksi_satu_langkah(X)
        keluar[:, t] = p
        X = np.hstack([X[:, 1:], p.reshape(-1, 1)])
    return keluar

def latih_lag(y, n_train):
    X = np.array([y[i:i+W] for i in range(n_train - W)])
    t = np.array([y[i+W] for i in range(n_train - W)])
    return X, t
print('fungsi protokol siap')

## 4. Definisi model

ARIMA memilih orde lewat pencarian kisi berbasis AIC **sekali** pada bagian latih,
lalu memakai `apply()` untuk memperbarui keadaan di tiap origin tanpa melatih
ulang parameternya.

Jaringan saraf memakai jendela masukan yang sama (12 langkah) agar perbandingan
adil. Normalisasi memakai statistik **bagian latih saja**, bukan seluruh deret —
memakai seluruh deret akan membocorkan informasi masa depan.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from statsmodels.tsa.arima.model import ARIMA

def pilih_orde_arima(y_train, pmax=2, dmax=1, qmax=2):
    terbaik, orde = np.inf, (1, 0, 0)
    for p, d, q in itertools.product(range(pmax+1), range(dmax+1), range(qmax+1)):
        if p == 0 and q == 0: continue
        try:
            aic = ARIMA(y_train, order=(p, d, q)).fit().aic
            if np.isfinite(aic) and aic < terbaik: terbaik, orde = aic, (p, d, q)
        except Exception: continue
    return orde

def bangun_dl(jenis, seed):
    from tensorflow import keras
    keras.utils.set_random_seed(seed)
    if jenis == 'MLP':
        m = keras.Sequential([keras.layers.Input((W,)), keras.layers.Dense(32, activation='relu'),
                              keras.layers.Dense(1)])
    elif jenis == 'DNN':
        m = keras.Sequential([keras.layers.Input((W,)), keras.layers.Dense(64, activation='relu'),
                              keras.layers.Dropout(0.2), keras.layers.Dense(32, activation='relu'),
                              keras.layers.Dense(1)])
    elif jenis == 'LSTM':
        m = keras.Sequential([keras.layers.Input((W, 1)), keras.layers.LSTM(32), keras.layers.Dense(1)])
    else:
        m = keras.Sequential([keras.layers.Input((W, 1)), keras.layers.GRU(32), keras.layers.Dense(1)])
    m.compile(optimizer='adam', loss='mse')
    return m
print('definisi model siap')

## 5. Jalankan evaluasi

Sel ini adalah bagian terberat (±10 menit). Kemajuannya dicetak per variabel.

In [ ]:
hasil, galat_simpan = [], {}

for k, (nama, sat) in VAR.items():
    y = seri[k].values.astype(float)
    n = len(y); n_train = int(n * TRAIN_FRAC)
    origins = buat_origin(n, n_train, max(HORIZON))
    log(f'{nama}: n={n} latih={n_train} origin={len(origins)}')

    aktual = {h: np.array([y[o:o+h] for o in origins]) for h in HORIZON}
    X0 = jendela(y, origins); pred = {}

    for h in HORIZON:
        pred[('Naive', h)] = np.repeat(X0[:, -1:], h, axis=1)
        pred[('SeasonalNaive', h)] = np.array([[y[o + t - MUSIM] for t in range(h)] for o in origins])

    Xtr, ttr = latih_lag(y, n_train)
    lr = LinearRegression().fit(Xtr, ttr)
    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1).fit(Xtr, ttr)
    for h in HORIZON:
        pred[('Linear', h)] = rekursif(lambda X: lr.predict(X), X0, h)
        pred[('RandomForest', h)] = rekursif(lambda X: rf.predict(X), X0, h)
    log('  dasar + Linear + RF selesai')

    orde = pilih_orde_arima(y[:n_train])
    res0 = ARIMA(y[:n_train], order=orde).fit()
    ap = {h: np.zeros((len(origins), h)) for h in HORIZON}
    for i, o in enumerate(origins):
        try:    f = res0.apply(y[:o]).forecast(max(HORIZON))
        except Exception: f = np.repeat(y[o-1], max(HORIZON))
        for h in HORIZON: ap[h][i] = f[:h]
    for h in HORIZON: pred[('ARIMA', h)] = ap[h]
    log(f'  ARIMA orde={orde} selesai')

    mn, mx = float(Xtr.min()), float(Xtr.max()); rng = (mx - mn) or 1.0
    nz = lambda a: (a - mn) / rng
    dz = lambda a: a * rng + mn
    if MODEL_DL:
        from tensorflow import keras
    for jenis in MODEL_DL:
        kump = {h: [] for h in HORIZON}
        for sd in SEEDS:
            m = bangun_dl(jenis, sd)
            Xf = nz(Xtr).reshape(-1, W, 1) if jenis in ('LSTM', 'GRU') else nz(Xtr)
            m.fit(Xf, nz(ttr), epochs=EPOCHS, batch_size=32, verbose=0, validation_split=0.2,
                  callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)])
            def satu(X, mm=m, j=jenis):
                a = nz(X).reshape(-1, W, 1) if j in ('LSTM', 'GRU') else nz(X)
                return dz(mm.predict(a, verbose=0).ravel())
            for h in HORIZON: kump[h].append(rekursif(satu, X0, h))
            keras.backend.clear_session()
        for h in HORIZON: pred[(jenis, h)] = np.mean(kump[h], axis=0)
        log(f'  {jenis} selesai ({len(SEEDS)} benih)')

    skala = np.mean(np.abs(np.diff(y[:n_train])))   # skala MASE
    for (mdl, h), P in pred.items():
        g = aktual[h] - P
        galat_simpan[(k, mdl, h)] = g.ravel()
        hasil.append({'Variabel': nama, 'Model': mdl, 'Horizon': h,
                      'MAE': float(np.mean(np.abs(g))),
                      'RMSE': float(np.sqrt(np.mean(g**2))),
                      'MASE': float(np.mean(np.abs(g)) / skala)})

tab_hasil = pd.DataFrame(hasil)
log('evaluasi selesai')

In [ ]:
print('TABEL 2 — MAE per model x horizon')
for k, (nama, sat) in VAR.items():
    piv = tab_hasil[tab_hasil.Variabel == nama].pivot(index='Model', columns='Horizon', values='MAE')
    print(f'\n{nama} ({sat}):')
    display(piv.round(3))

## 6. Uji Diebold-Mariano

Inilah penentu kesimpulan. Uji ini memeriksa apakah selisih galat dua model
berbeda dari nol secara statistik, dengan ragam jangka panjang Newey-West
(bobot Bartlett) karena galat peramalan multi-langkah **pasti berautokorelasi**.

Dipakai koreksi sampel kecil **Harvey, Leybourne & Newbold (1997)**, sebab tanpa
koreksi itu uji DM terlalu sering menolak pada sampel terbatas.

In [ ]:
def dm(e1, e2, h, loss='squared'):
    e1, e2 = np.asarray(e1, float), np.asarray(e2, float)
    n = min(len(e1), len(e2))
    if n < 10: return np.nan, np.nan
    L = (lambda e: e**2) if loss == 'squared' else (lambda e: np.abs(e))
    d = L(e1[:n]) - L(e2[:n]); db = float(np.mean(d)); dc = d - db
    lrv = float(np.mean(dc**2))
    for kk in range(1, h):
        if kk >= n: break
        lrv += 2.0 * (1.0 - kk/h) * float(np.mean(dc[kk:] * dc[:-kk]))
    if lrv <= 0: return np.nan, np.nan
    stat = db / math.sqrt(lrv / n)
    koreksi = math.sqrt(max((n + 1 - 2*h + h*(h-1)/n) / n, 1e-12))
    s = stat * koreksi
    return s, 2 * (1 - sstats.t.cdf(abs(s), df=n-1))

baris = []
for k, (nama, _) in VAR.items():
    for h in HORIZON:
        models = sorted({m for (kk, m, hh) in galat_simpan if kk == k and hh == h})
        for a, b in itertools.combinations(models, 2):
            s, p = dm(galat_simpan[(k, a, h)], galat_simpan[(k, b, h)], h)
            baris.append({'Variabel': nama, 'Horizon': h, 'Model A': a, 'Model B': b,
                          'DM': round(s, 3) if np.isfinite(s) else np.nan,
                          'p': round(p, 4) if np.isfinite(p) else np.nan,
                          'Lebih baik': (a if s < 0 else b) if np.isfinite(p) and p < 0.05 else 'tidak berbeda'})
tab_dm = pd.DataFrame(baris)
print(f'TABEL 3 — Diebold-Mariano ({len(tab_dm)} pasangan)')
display(tab_dm.head(15))

## 7. Bootstrap blok untuk selang kepercayaan MAE

Bootstrap biasa mengandaikan pengamatan saling bebas — keliru untuk galat
peramalan. **Bootstrap blok bergerak** mengambil ulang potongan berurutan
sehingga struktur autokorelasi di dalam blok tetap terjaga.

In [ ]:
def boot_ci(e, n_boot=N_BOOT, blok=None, alpha=0.05, seed=42):
    e = np.asarray(e, float); n = len(e)
    blok = blok or max(2, int(np.sqrt(n)))
    if n < blok*2: return (np.nan, np.nan)
    g = np.random.default_rng(seed); nb = int(np.ceil(n/blok)); maks = n - blok
    v = np.empty(n_boot)
    for i in range(n_boot):
        st = g.integers(0, maks+1, size=nb)
        v[i] = np.mean(np.abs(np.concatenate([e[s:s+blok] for s in st])[:n]))
    return float(np.percentile(v, 100*alpha/2)), float(np.percentile(v, 100*(1-alpha/2)))

tab_hasil['MAE_lo'], tab_hasil['MAE_hi'] = np.nan, np.nan
for idx, r in tab_hasil.iterrows():
    kk = [x for x in VAR if VAR[x][0] == r['Variabel']][0]
    lo, hi = boot_ci(galat_simpan[(kk, r['Model'], r['Horizon'])])
    tab_hasil.at[idx, 'MAE_lo'], tab_hasil.at[idx, 'MAE_hi'] = lo, hi
log('bootstrap selesai')
display(tab_hasil.head(12).round(3))

## 8. Juara per horizon — dan apakah kemenangannya nyata

Dua pertanyaan dipisahkan dengan tegas:

1. **Siapa ber-MAE terendah?** — mudah dijawab, tetapi bisa menyesatkan.
2. **Apakah selisihnya signifikan?** — inilah yang boleh dilaporkan sebagai temuan.

Kolom terakhir menjawab pertanyaan kedua. Bila tertulis *tidak berbeda*, maka
model yang MAE-nya terendah **tidak boleh diklaim lebih baik**.

In [ ]:
def cari_dm(var, h, a, b):
    m = tab_dm[(tab_dm.Variabel == var) & (tab_dm.Horizon == h) &
               (((tab_dm['Model A'] == a) & (tab_dm['Model B'] == b)) |
                ((tab_dm['Model A'] == b) & (tab_dm['Model B'] == a)))]
    return m.iloc[0] if len(m) else None

baris = []
for k, (nama, _) in VAR.items():
    sub = tab_hasil[tab_hasil.Variabel == nama]
    for h in HORIZON:
        s = sub[sub.Horizon == h].sort_values('MAE')
        juara = s.iloc[0]['Model']
        mae_naive = float(sub[(sub.Horizon == h) & (sub.Model == 'Naive')]['MAE'].iloc[0])
        if juara == 'Naive':
            vs_naive, p_naive = '(juara itu sendiri)', np.nan
        else:
            r = cari_dm(nama, h, juara, 'Naive')
            vs_naive, p_naive = r['Lebih baik'], r['p']
        if juara in ('Naive', 'Linear'):
            vs_lin, p_lin = '-', np.nan
        else:
            r2 = cari_dm(nama, h, juara, 'Linear')
            vs_lin, p_lin = r2['Lebih baik'], r2['p']
        baris.append({'Variabel': nama, 'Horizon': h, 'Juara (MAE)': juara,
                      'MAE': round(float(s.iloc[0]['MAE']), 3), 'MAE Naive': round(mae_naive, 3),
                      'vs Naive': vs_naive, 'p (vs Naive)': p_naive,
                      'vs Linear': vs_lin, 'p (vs Linear)': p_lin})
tab_juara = pd.DataFrame(baris)
print('TABEL 4 — Juara per horizon beserta uji signifikansinya')
display(tab_juara)

## 9. Grafik MAE terhadap horizon

Pita menunjukkan selang kepercayaan 95%. **Pita yang bertumpang tindih berarti
selisihnya belum tentu nyata** — sesuai dengan hasil uji Diebold-Mariano.

In [ ]:
SOROT = ['Naive', 'SeasonalNaive', 'Linear', 'ARIMA', 'MLP', 'LSTM', 'GRU']
fig, axes = plt.subplots(1, len(VAR), figsize=(16, 4.4))
for ax, (k, (nama, sat)) in zip(np.atleast_1d(axes), VAR.items()):
    sub = tab_hasil[tab_hasil.Variabel == nama]
    for mdl in SOROT:
        d = sub[sub.Model == mdl].sort_values('Horizon')
        if d.empty: continue
        ax.plot(d.Horizon, d.MAE, marker='o', label=mdl, linewidth=1.8)
        ax.fill_between(d.Horizon, d.MAE_lo, d.MAE_hi, alpha=0.12)
    ax.set_title(f'{nama} ({sat})'); ax.set_xlabel('Horizon (langkah, 1 = 5 menit)')
    ax.set_ylabel('MAE'); ax.grid(alpha=.3); ax.set_xticks(HORIZON)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT}/mae_vs_horizon.png', dpi=150); plt.show()

## 10. Simpulan dan ekspor

Sel ini merangkum bukti menjadi pernyataan yang boleh dibawa ke naskah.
Perhatikan bahwa pernyataan hanya dibuat bila **didukung uji**, bukan sekadar
selisih MAE.

In [ ]:
rangkum = []
for k, (nama, _) in VAR.items():
    j = tab_juara[tab_juara.Variabel == nama]
    menang_naive = j[(j['p (vs Naive)'] < 0.05)]
    unggul_dl = j[(j['p (vs Linear)'] < 0.05)]
    if len(menang_naive) == 0:
        simpul = 'Tidak ada model yang signifikan mengalahkan Naive'
    elif len(unggul_dl) == 0:
        simpul = (f'Model signifikan mengalahkan Naive pada {len(menang_naive)}/{len(j)} horizon, '
                  'tetapi jaringan saraf TIDAK signifikan mengungguli Linear')
    else:
        simpul = f'Jaringan saraf signifikan mengungguli Linear pada {len(unggul_dl)}/{len(j)} horizon'
    rangkum.append({'Variabel': nama,
                    'Horizon dengan model > Naive (signifikan)': f'{len(menang_naive)}/{len(j)}',
                    'Horizon dengan DL > Linear (signifikan)': f'{len(unggul_dl)}/{len(j)}',
                    'Simpulan': simpul})
tab_simpul = pd.DataFrame(rangkum)
print('TABEL 5 — Simpulan berbasis uji'); display(tab_simpul)

import shutil
for nm, t in (('tabel1_praproses', tab_pra), ('tabel2_hasil', tab_hasil),
              ('tabel3_diebold_mariano', tab_dm), ('tabel4_juara', tab_juara),
              ('tabel5_simpulan', tab_simpul)):
    t.to_csv(f'{OUT}/{nm}.csv', index=False)
shutil.make_archive('hasil_fase2', 'zip', OUT)
print('\nberkas:', sorted(os.listdir(OUT)))
try:
    from google.colab import files; files.download('hasil_fase2.zip')
except Exception:
    print('(unduhan otomatis hanya di Colab)')